# Heart Disease EDA & Predictive Analysis
**Author:** Swananda  
**Dataset:** UCI Heart Disease (Cleveland)  
**Objective:** Perform comprehensive Exploratory Data Analysis on the Cleveland Heart Disease dataset and extract actionable insights.

---

## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Output directory for saved figures
os.makedirs('screenshots', exist_ok=True)

# Load cleaned CSV
df = pd.read_csv('heart_disease_cleveland_cleaned.csv')
print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')

## 2. Dataset Summary

In [ ]:
print('=== Column Data Types ===')
print(df.dtypes)
print()
print('=== First 5 Rows ===')
df.head()

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(2)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
print('=== Target Distribution ===')
print(df['target'].value_counts())
print(f"\nDisease prevalence: {df['target'].mean()*100:.1f}%")

## 3. EDA — Histograms for Continuous Features

In [ ]:
BLUE  = '#2563EB'
RED   = '#DC2626'
TEAL  = '#0D9488'
ORANGE= '#EA580C'

cont_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cont_labels = {
    'age': 'Age (years)',
    'trestbps': 'Resting Blood Pressure (mm Hg)',
    'chol': 'Serum Cholesterol (mg/dl)',
    'thalach': 'Max Heart Rate Achieved',
    'oldpeak': 'ST Depression (oldpeak)'
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

colors = [BLUE, TEAL, ORANGE, RED, '#7C3AED', BLUE]

for i, col in enumerate(cont_cols):
    axes[i].hist(df[col], bins=25, color=colors[i], edgecolor='white', linewidth=0.6, alpha=0.9)
    axes[i].set_title(cont_labels[col], fontsize=12, fontweight='bold', pad=8)
    axes[i].set_xlabel(cont_labels[col], fontsize=9)
    axes[i].set_ylabel('Count', fontsize=9)
    axes[i].spines[['top', 'right']].set_visible(False)

axes[5].axis('off')
fig.suptitle('Distribution of Continuous Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('screenshots/01_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/01_histograms.png')

## 4. EDA — Bar Charts for Categorical Features

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
cat_labels = {
    'sex':     'Sex (0=F, 1=M)',
    'cp':      'Chest Pain Type',
    'fbs':     'Fasting Blood Sugar >120',
    'restecg': 'Resting ECG',
    'exang':   'Exercise Induced Angina',
    'slope':   'Slope of ST Segment',
    'ca':      'No. of Major Vessels (CA)',
    'thal':    'Thalassemia Type'
}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
bar_colors = [BLUE, TEAL, ORANGE, RED, '#7C3AED', TEAL, ORANGE, BLUE]

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts().sort_index()
    axes[i].bar(counts.index.astype(str), counts.values, color=bar_colors[i],
                edgecolor='white', linewidth=0.6, width=0.55)
    axes[i].set_title(cat_labels[col], fontsize=11, fontweight='bold', pad=6)
    axes[i].set_xlabel('Category', fontsize=9)
    axes[i].set_ylabel('Count', fontsize=9)
    for bar in axes[i].patches:
        axes[i].annotate(f'{int(bar.get_height())}',
                         (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                         ha='center', va='bottom', fontsize=8, fontweight='bold')
    axes[i].spines[['top', 'right']].set_visible(False)

fig.suptitle('Distribution of Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/02_barcharts_categorical.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/02_barcharts_categorical.png')

## 5. EDA — Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

target_counts = df['target'].value_counts().sort_index()
bar_c = [BLUE, RED]
axes[0].bar(['No Disease (0)', 'Disease (1)'], target_counts.values,
            color=bar_c, edgecolor='white', width=0.5)
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height())}',
                     (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_title('Target Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].pie(target_counts.values, labels=['No Disease (0)', 'Disease (1)'],
            autopct='%1.1f%%', colors=bar_c, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Target Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('screenshots/03_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/03_target_distribution.png')

## 6. EDA — Comparative Charts (Target vs Key Features)

In [ ]:
comp_cols = ['sex', 'cp', 'thal', 'ca']
comp_labels = {
    'sex': 'Sex (0=Female, 1=Male)',
    'cp':  'Chest Pain Type',
    'thal':'Thalassemia Type',
    'ca':  'No. of Major Vessels (CA)'
}

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for i, col in enumerate(comp_cols):
    pivot = df.groupby([col, 'target']).size().unstack(fill_value=0)
    pivot.columns = ['No Disease', 'Disease']
    x = np.arange(len(pivot))
    w = 0.38
    axes[i].bar(x - w/2, pivot['No Disease'], width=w, label='No Disease (0)',
                color=BLUE, edgecolor='white')
    axes[i].bar(x + w/2, pivot['Disease'],    width=w, label='Disease (1)',
                color=RED,  edgecolor='white')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(pivot.index.astype(str))
    axes[i].set_title(f'Target vs {comp_labels[col]}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col.upper(), fontsize=9)
    axes[i].set_ylabel('Count', fontsize=9)
    axes[i].legend(fontsize=8)
    axes[i].spines[['top', 'right']].set_visible(False)

fig.suptitle('Target vs Key Clinical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/04_comparative_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/04_comparative_charts.png')

## 7. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('screenshots/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/05_correlation_heatmap.png')

## 8. Feature Correlations with Target

In [ ]:
target_corr = df.corr(numeric_only=True)['target'].drop('target').sort_values()

colors_bar = [RED if v > 0 else BLUE for v in target_corr.values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='white', height=0.6)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Feature Correlation with Target (Heart Disease)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

for bar in ax.patches:
    xval = bar.get_width()
    ax.text(xval + 0.005 if xval >= 0 else xval - 0.005,
            bar.get_y() + bar.get_height() / 2,
            f'{xval:.2f}', va='center', fontsize=8,
            ha='left' if xval >= 0 else 'right')

plt.tight_layout()
plt.savefig('screenshots/06_target_correlation_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/06_target_correlation_bar.png')
print(target_corr)

## 9. Age Distribution by Target

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for val, color, label in zip([0, 1], [BLUE, RED], ['No Disease', 'Disease']):
    ax.hist(df[df['target'] == val]['age'], bins=20, alpha=0.7,
            color=color, label=label, edgecolor='white')
ax.set_title('Age Distribution by Heart Disease Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('screenshots/07_age_by_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/07_age_by_target.png')

## 10. Boxplots — Continuous Features vs Target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
palette = {0: BLUE, 1: RED}

for i, col in enumerate(cont_cols):
    sns.boxplot(data=df, x='target', y=col, palette=palette, ax=axes[i],
                linewidth=1.2)
    axes[i].set_title(cont_labels[col], fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Target (0=No Disease, 1=Disease)', fontsize=8)
    axes[i].set_ylabel(col)
    axes[i].spines[['top', 'right']].set_visible(False)

fig.suptitle('Boxplots: Continuous Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/08_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/08_boxplots.png')

## 11. Key Insights Summary

In [ ]:
print('=' * 60)
print('         KEY INSIGHTS — HEART DISEASE EDA')
print('=' * 60)

print('\n1. TARGET BALANCE')
vc = df['target'].value_counts()
print(f'   No Disease (0): {vc[0]} ({vc[0]/len(df)*100:.1f}%)')
print(f'   Disease    (1): {vc[1]} ({vc[1]/len(df)*100:.1f}%)')

print('\n2. TOP FEATURES CORRELATED WITH DISEASE')
top = target_corr.abs().sort_values(ascending=False).head(5)
for f, v in top.items():
    print(f'   {f:12s}  |r| = {v:.3f}')

print('\n3. AGE')
print(f'   Mean age No Disease: {df[df.target==0].age.mean():.1f}')
print(f'   Mean age Disease:    {df[df.target==1].age.mean():.1f}')

print('\n4. SEX DISTRIBUTION')
sex_tgt = df.groupby('sex')['target'].mean()
print(f'   Females (0): {sex_tgt[0]*100:.1f}% disease rate')
print(f'   Males   (1): {sex_tgt[1]*100:.1f}% disease rate')

print('\n5. CA (Major Vessels)')
ca_tgt = df.groupby('ca')['target'].mean()
print(ca_tgt.to_string())

print('\n6. THAL (Thalassemia)')
thal_tgt = df.groupby('thal')['target'].mean()
print(thal_tgt.to_string())

print('\n7. CHEST PAIN TYPE vs DISEASE')
cp_tgt = df.groupby('cp')['target'].mean()
print(cp_tgt.to_string())

print('\n' + '=' * 60)